# Company

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier,RandomForestRegressor, AdaBoostClassifier,AdaBoostRegressor
from catboost import CatBoostClassifier
from xgboost import XGBRegressor,XGBClassifier


from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report


In [ ]:
company= pd.read_csv("Company_Data.csv")
company.head()

,Sales,CompPrice,Income,Advertising,Population,Price,ShelveLoc,Age,Education,Urban,US
0,9.50,138,73,11,276,120,Bad,42,17,Yes,Yes
1,11.22,111,48,16,260,83,Good,65,10,Yes,Yes
2,10.06,113,35,10,269,80,Medium,59,12,Yes,Yes
3,7.40,117,100,4,466,97,Medium,55,14,Yes,Yes
4,4.15,141,64,3,340,128,Bad,38,13,Yes,No


In [ ]:
company['Sales_Category'] = pd.cut(company['Sales'], bins=[0, 5, 10, 15, 20, 25], labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])

In [ ]:
for col_name in company.columns:
    if(company[col_name].dtype == 'object'):
        company[col_name]= company[col_name].astype('category')
        company[col_name] = company[col_name].cat.codes

In [ ]:
company=company.dropna()

In [ ]:
X = company.drop(['Sales', 'Sales_Category'], axis=1)
y = company['Sales_Category']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [ ]:
feature_importances = pd.DataFrame({'feature': X.columns, 'importance': rf_model.feature_importances_})
feature_importances = feature_importances.sort_values(by='importance', ascending=False)


In [ ]:
print(feature_importances)


       feature  importance
4        Price    0.202259
6          Age    0.125242
0    CompPrice    0.122612
3   Population    0.122319
1       Income    0.113485
5    ShelveLoc    0.111475
2  Advertising    0.096239
7    Education    0.071567
9           US    0.019991
8        Urban    0.014811


In [ ]:
accuracy = rf_model.score(X_test, y_test)
print("Accuracy:", accuracy)

Accuracy: 0.7375


In [ ]:
y_pred = rf_model.predict(X_test)
print("Classification Report:")
print(classification_report(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

         Low       0.72      0.96      0.82        50
      Medium       0.83      0.31      0.45        16
    Very Low       0.86      0.43      0.57        14

    accuracy                           0.74        80
   macro avg       0.80      0.57      0.62        80
weighted avg       0.76      0.74      0.70        80



# Stacking classifier

In [ ]:
from sklearn.ensemble import StackingClassifier,StackingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

In [ ]:
# Define base estimators
base_estimators = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)), #bagging
    ('ada', AdaBoostClassifier(n_estimators=100, random_state=42)),    #boosting
    ('xgb', XGBClassifier()), #boosting
    ('catboost', CatBoostClassifier())  #boosting
]


In [ ]:
# Define meta classifier
meta_estimator = LogisticRegression()

In [ ]:
# Initialize Stacking Classifier
stacking_model = StackingClassifier(estimators=base_estimators, final_estimator=meta_estimator)

# Fit the Stacking Classifier
stacking_model.fit(X_train, y_train)


Learning rate set to 0.074648
0:	learn: 1.3280514	total: 50.4ms	remaining: 50.3s
1:	learn: 1.2840197	total: 53.4ms	remaining: 26.6s
2:	learn: 1.2559680	total: 56.3ms	remaining: 18.7s
3:	learn: 1.2271224	total: 59ms	remaining: 14.7s
4:	learn: 1.1776325	total: 62.1ms	remaining: 12.4s
5:	learn: 1.1501624	total: 65ms	remaining: 10.8s
6:	learn: 1.1173216	total: 68.1ms	remaining: 9.66s
7:	learn: 1.0897623	total: 71.4ms	remaining: 8.86s
8:	learn: 1.0591946	total: 74.6ms	remaining: 8.21s
9:	learn: 1.0345973	total: 77.4ms	remaining: 7.67s
10:	learn: 1.0110790	total: 82ms	remaining: 7.37s
11:	learn: 0.9891838	total: 84.9ms	remaining: 6.99s
12:	learn: 0.9706675	total: 87.8ms	remaining: 6.66s
13:	learn: 0.9536944	total: 90.7ms	remaining: 6.39s
14:	learn: 0.9362893	total: 93.5ms	remaining: 6.14s
15:	learn: 0.9201828	total: 96.4ms	remaining: 5.93s
16:	learn: 0.9062567	total: 99.3ms	remaining: 5.74s
17:	learn: 0.8909856	total: 102ms	remaining: 5.57s
18:	learn: 0.8725724	total: 105ms	remaining: 5.42s


/usr/local/lib/python3.10/dist-packages/sklearn/model_selection/_split.py:700: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/model_selection/_split.py:700: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/model_selection/_split.py:700: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/model_selection/_split.py:700: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


Streaming output truncated to the last 5000 lines.
4:	learn: 1.2002505	total: 13.2ms	remaining: 2.63s
5:	learn: 1.1677863	total: 15.8ms	remaining: 2.62s
6:	learn: 1.1447482	total: 18.3ms	remaining: 2.59s
7:	learn: 1.1211234	total: 20.9ms	remaining: 2.59s
8:	learn: 1.0989007	total: 23.3ms	remaining: 2.57s
9:	learn: 1.0762098	total: 25.9ms	remaining: 2.56s
10:	learn: 1.0498065	total: 28.5ms	remaining: 2.56s
11:	learn: 1.0325163	total: 31ms	remaining: 2.55s
12:	learn: 1.0143340	total: 33.4ms	remaining: 2.54s
13:	learn: 0.9965328	total: 35.9ms	remaining: 2.53s
14:	learn: 0.9773360	total: 38.3ms	remaining: 2.52s
15:	learn: 0.9601657	total: 40.8ms	remaining: 2.51s
16:	learn: 0.9439614	total: 43.2ms	remaining: 2.5s
17:	learn: 0.9265829	total: 45.8ms	remaining: 2.5s
18:	learn: 0.9114158	total: 48.3ms	remaining: 2.49s
19:	learn: 0.8975162	total: 50.8ms	remaining: 2.49s
20:	learn: 0.8864868	total: 53.5ms	remaining: 2.5s
21:	learn: 0.8710601	total: 56.2ms	remaining: 2.5s
22:	learn: 0.8571629	tota

StackingClassifier(estimators=[('rf', RandomForestClassifier(random_state=42)),
                               ('ada',
                                AdaBoostClassifier(n_estimators=100,
                                                   random_state=42)),
                               ('xgb',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=None,
                                              device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric=Non...
                                              max_cat_threshold=None,
                                              max_cat_to_onehot=None,
                                              max_delta_step=None,
                                              max_depth=None, max_leaves=None,
                                              min_child_weight=None,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=None, n_jobs=None,
                                              num_parallel_tree=None,
                                              random_state=None, ...)),
                               ('catboost',
                                <catboost.core.CatBoostClassifier object at 0x7be0ebea08e0>)],
                   final_estimator=LogisticRegression())

In [ ]:
# Evaluate the Stacking Classifier
accuracy = stacking_model.score(X_test, y_test)
print("Accuracy:", accuracy)

y_pred = stacking_model.predict(X_test)
print("Classification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.8
Classification Report:
              precision    recall  f1-score   support

         Low       0.78      0.94      0.85        50
      Medium       0.77      0.62      0.69        16
    Very Low       1.00      0.50      0.67        14

    accuracy                           0.80        80
   macro avg       0.85      0.69      0.74        80
weighted avg       0.82      0.80      0.79        80

